#### Tworzymy reżyserów przez panel administratora

Możemy już tworzyć reżyserów z panelu administracyjnego.

Stwórzmy jeszcze kilku reżyserów i przejdźmy do widoku szczegółów jakiegoś filmu.

Możemy teraz do filmu przypisać reżysera przy pomocy listy rozwijanej. A dokładniej musimy przypisać przynajmniej
jednego, ponieważ nie oznaczyliśmy pola reżysera jako `blank=True` co by pozwalało na pozostawienie go pustym.

# Django - Rozbudowa modelu danych 2
*[Mikołaj Leszczuk](mailto:mikolaj.leszczuk@agh.edu.pl), [Agnieszka Rudnicka](mailto:rudnicka@agh.edu.pl)*

* Ulepszamy nasze modele
  * Recenzje filmów
    * Model recenzji
    * Migracje modelu recenzji
* Nowe widoki dla użytkowników
  * Widoki klasowe
  * Przykładowy widok klasowy
  * Widok klasowy listy filmów
  * Podpinanie widoków klasowych w `urls.py`
* Dalsze prace

## Ulepszamy nasze modele

### Recenzje filmów

Mamy już model ([`movies/models.py`](http://localhost:8888/edit/movies/models.py)) reżysera i filmu, które są ze sobą powiązane. Pora na model recenzji oraz widoki szczegółów, które
wyświetlą te dane użytkownikom. W tej chwili tylko administratorzy je widzą i mogą modyfikować.

#### Model recenzji

W pliku [`movies/models.py`](http://localhost:8888/edit/movies/models.py)

```python
class Review(models.Model):
    movie = models.ForeignKey(to="movies.Movie", verbose_name="recenzja filmu", on_delete=models.CASCADE)
    author = models.CharField(verbose_name="autor recenzji", max_length=250)
    content = models.TextField(verbose_name="treść recenzji")
    is_recommended = models.BooleanField(verbose_name="polecam innym")

    class Meta:
        verbose_name = "recenzja"
        verbose_name_plural = "recenzje"
```

Model zawiera:

* relację (konkretniej klucz obcy) do filmu;

* pole tekstowe na podpis użytkownika-autora recenzji;

* pole tekstowe na recenzję;

* pole prawda/fałsz, które domyślnie będzie reprezentowane przez checkbox, czy użytkownik poleca dany film.

#### Migracje modelu recenzji

Następnym krokiem jest wygenerowanie migracji i zaaplikowanie jej. Nie zapomnijmy również dodać do panelu
administracyjnego ([`movies/admin.py`](http://localhost:8888/edit/movies/admin.py))!

In [9]:
!python3 manage.py makemigrations

No changes detected


In [10]:
!python3 manage.py migrate

Operations to perform:
  Apply all migrations: admin, auth, contenttypes, movies, sessions
Running migrations:
  No migrations to apply.


```python
from movies.models import Movie, Director, Review

admin.site.register(Review)
```

## Nowe widoki dla użytkowników

### Widoki klasowe

Mechanizmem, którego do tej pory nie używaliśmy są widoki klasowe. W Internecie można je znaleźć pod nazwą "Class
Based Views".

Polecam stronę z przeglądem widoków klasowych: [https://ccbv.co.uk/](https://ccbv.co.uk/) a także dokumentację: [https://docs.djangoproject.com/en/5.2/topics/class-based-views/intro/](https://docs.djangoproject.com/en/5.2/topics/class-based-views/intro/)

W skrócie - jest to podejście do pisania widoków z wykorzystaniem klas. Podobnie jak klasy definiują nam modele, tutaj klasy definiują widoki. Jest to proste i wyręcza nas z pisania części powtarzalnego kodu.

#### Przykładowy widok klasowy

Załóżmy, że mamy widok, który w zależności od metody zwraca 2 różne rzeczy (podobnie jak było z formularzem rejestracji na zajęciach o użytkownikach).

```python
from django.http import HttpResponse

def my_view(request):
    if request.method == 'GET':
        # ...
        return HttpResponse('result')
    if request.method == 'POST':
        # ...
        return HttpResponse('another result')
```

Gdybyśmy wykorzystali widoki klasowe, to można to zapisać tak jak poniżej.

```python
from django.http import HttpResponse
from django.views import View

class MyView(View):
    def get(self, request):
        # ...
        return HttpResponse('result')

    def post(self, request):
        # ...
        return HttpResponse('another result')
```

Na pierwszy rzut oka może nie wydawać się to krótsze, lub w jakikolwiek sposób lepsze. Zauważmy jednak, że zamiast
pisać jedną dłuższą funkcję, która kolejno sprawdza typ metody zapytania i potem wykonuje kod, tworzymy jedynie klasę z dwoma metodami (`def get()` oraz `def post()`), które zawierają tylko ten kod, który dotyczy danego przypadku.

#### Widok klasowy listy filmów

Może następny przykład będzie bardziej przekonujący, oto lista filmów dotychczas ([`movies/views.py`](http://localhost:8888/edit/movies/views.py), [`python3 manage.py runserver`](/terminals/1), [http://127.0.0.1:8000/filmy](http://127.0.0.1:8000/filmy)):

```python
from django.shortcuts import render
from movies.models import Movie

def movie_list(request):
    movies = Movie.objects.all()
    context = {
        "movies": movies,
    }
    return render(request, template_name="movie_list.html", context=context)
```

A to lista filmów jako widok klasowy:

```python
from django.views.generic import ListView
from .models import Movie

class MovieListView(ListView):
    model = Movie
```

*Et voilà!*

Django wyciągnie domyślnie wszystkie filmy, bo podaliśmy model `Movie` i poszuka szablony HTML na podstawie nazwy
tego modelu. W tym przypadku będzie to `Movie_list.html`.

#### Alternatywny szablon do widoku klasowego

Aby zrozumieć działanie `ListView`, przygotujemy osobny szablon o nazwie [`movies/templates/movie_list_v2.html`](http://localhost:8888/edit/movies/templates/movie_list_v2.html), który korzysta z domyślnego kontekstu `object_list` przekazywanego przez Django.

In [11]:
!touch movies/templates/movie_list_v2.html

Dzięki temu nie musimy zmieniać istniejącego `movie_list.html` i można porównać działanie obu podejść.

[`movies/templates/movie_list.html`](http://localhost:8888/edit/movies/templates/movie_list.html):

```django
{{ movies }}

{% for movie in movies %}
<p>
    Film: "{{ movie }}"
</p>
{% endfor %}
```

[`movies/templates/movie_list_v2.html`](http://localhost:8888/edit/movies/templates/movie_list_v2.html):

```django
<h1>Lista filmów (widok klasowy)</h1>

<ul>
{% for movie in object_list %}
    <li>{{ movie }}</li>
{% endfor %}
</ul>
```

Skoro nasz szablon nazywa się inaczej niż klasa, możemy podać dodatkowe pole `template_name`, o tak:

```python
class MovieListView(ListView):
    model = Movie
    template_name = "movie_list_v2.html"
```

#### Podpinanie widoków klasowych w `urls.py`

Aby wykorzystać taki widok klasowy z [`goodmovies/urls.py`](http://localhost:8888/edit/goodmovies/urls.py) musimy go zaimportować a następnie wywołać metodę `as_view()`.

```python
from movies.views import MovieListView

urlpatterns += [
    path('about/', MovieListView.as_view()),
]
```

#### Testowanie `/about/`

Uruchom serwer deweloperski ([`python3 manage.py runserver`](/terminals/1)) i wejdź na stronę [http://127.0.0.1:8000/about/](http://127.0.0.1:8000/about/).  
Powinna wyświetlić się poprawna lista filmów z widokiem klasowym.

## Dalsze prace

W następnych krokach zróbmy rzeczy, które już umiemy:

* widok listy reżyserów;

* widok listy recenzji;

* szczegółowy widok filmu i reżysera

* linki, które pozwolą na przechodzenie między recenzjami, reżyserami i filmami.

### Widok listy reżyserów

Wyświetlimy listę wszystkich reżyserów zapisanych w bazie danych. Wykorzystamy klasowy widok `ListView`.

#### Plik: `views.py`

In [ ]:
from django.views.generic import ListView
from .models import Director

class DirectorListView(ListView):
    model = Director
    template_name = "director_list.html"

#### Plik: `urls.py`

In [ ]:
from movies.views import DirectorListView

urlpatterns += [
    path("rezyserzy/", DirectorListView.as_view()),
]

#### Szablon: `templates/director_list.html`

In [ ]:
<h1>Lista reżyserów</h1>

<ul>
{% for director in object_list %}
    <li>{{ director.last_name }}</li>
{% endfor %}
</ul>

### Widok listy recenzji

Podobnie jak z filmami i reżyserami, stworzymy widok, który wyświetli wszystkie recenzje przechowywane w bazie danych.

#### Plik: `views.py`

In [ ]:
from .models import Review

class ReviewListView(ListView):
    model = Review
    template_name = "review_list.html"

#### Plik: `urls.py`

In [ ]:
from movies.views import ReviewListView

urlpatterns += [
    path("recenzje/", ReviewListView.as_view()),
]

#### Szablon: `templates/review_list.html`

In [ ]:
<h1>Recenzje</h1>

<ul>
{% for review in object_list %}
    <li>
        {{ review.author }}: {{ review.content }} (film: {{ review.movie.title }})
    </li>
{% endfor %}
</ul>

#### Co oznacza `review.movie.title`?

To zapis dostępu do powiązanych obiektów w modelu Django. Mówi on:
- `review` – aktualna recenzja,
- `movie` – obiekt filmu powiązany z tą recenzją (ForeignKey),
- `title` – tytuł tego filmu.

Django pozwala w ten sposób nawigować po relacjach między modelami w szablonach HTML.

### Szczegółowy widok filmu i reżysera

#### Plik: `views.py`

In [ ]:
from django.views.generic import DetailView

class MovieDetailView(DetailView):
    model = Movie
    template_name = "movie_detail.html"

class DirectorDetailView(DetailView):
    model = Director
    template_name = "director_detail.html"

#### Co to jest `DetailView`?

`DetailView` to wbudowany widok klasowy Django, który automatycznie wyświetla szczegóły pojedynczego obiektu (np. konkretnego filmu lub reżysera).

Działa podobnie jak `ListView`, ale dla jednego obiektu. Wymaga podania `model` i `template_name`, a w adresie URL najczęściej przekazuje się `pk` (primary key), czyli identyfikator obiektu.

#### Plik: `urls.py`

In [ ]:
from movies.views import MovieDetailView, DirectorDetailView

urlpatterns += [
    path("film/<int:pk>/", MovieDetailView.as_view(), name="movie-detail"),
    path("rezyser/<int:pk>/", DirectorDetailView.as_view(), name="director-detail"),
]

#### Co oznacza `name="movie-detail"`?

Parametr `name` w ścieżce URL służy do jednoznacznego identyfikowania danego widoku w całej aplikacji. Dzięki temu można później odwoływać się do tego widoku w szablonach lub w kodzie Pythona za pomocą `{% url 'movie-detail' %}` zamiast wpisywać pełny adres URL.

To dobre praktyka, bo ułatwia późniejszą zmianę ścieżek bez konieczności modyfikowania szablonów.

#### Co oznacza `/<int:pk>/` w adresie URL?

To składnia Django do definiowania dynamicznych segmentów adresu URL. Oznacza:

- `int` – oczekiwany typ to liczba całkowita,
- `pk` – nazwa parametru, który będzie przekazany do widoku jako `pk` (primary key, czyli ID obiektu).

Przykład:

```python
path("film/<int:pk>/", MovieDetailView.as_view(), name="movie-detail")
```

Taki zapis pozwala odwiedzać różne filmy, np. `/film/1/`, `/film/42/` itp.

#### Szablon: `templates/movie_detail.html`

In [ ]:
<h1>{{ object.title }}</h1>
<p>Reżyser: <a href="{% url 'director-detail' object.director.id %}">{{ object.director.last_name }}</a></p>
<p>Opis: {{ object.short_description }}</p>

#### Szablon: `templates/director_detail.html`

In [ ]:
<h1>{{ object.last_name }}</h1>
<p>Filmy tego reżysera:</p>
<ul>
{% for movie in object.movies.all %}
    <li><a href="{% url 'movie-detail' movie.id %}">{{ movie.title }}</a></li>
{% endfor %}
</ul>

#### Co robi `{% url ... %}` w szablonach Django?

To specjalna instrukcja (template tag), która generuje adres URL na podstawie nazwy widoku (`name`) zdefiniowanej w `urls.py`.

Przykład:

```html
<a href="{% url 'director-detail' object.director.id %}">{{ object.director.last_name }}</a>
```

Wygeneruje link do widoku `director-detail` z odpowiednim ID filmu (np. `/rezyser/5/`).